# Week 2 — Day 1: Frontier APIs across providers

Connecting to OpenAI, Anthropic, Gemini, DeepSeek, Groq, Grok, OpenRouter, and Ollama — all through a single client (the OpenAI Python library), since most providers expose OpenAI-compatible endpoints. Then a tour of:
- Reasoning effort tuning
- Hard puzzle benchmarking across frontier models
- Prompt caching with LiteLLM
- An adversarial conversation between two chatbots

## API key setup

Add any of the following to your `.env`:
```
OPENAI_API_KEY=...
ANTHROPIC_API_KEY=...
GOOGLE_API_KEY=...
DEEPSEEK_API_KEY=...
GROQ_API_KEY=...
GROK_API_KEY=...
OPENROUTER_API_KEY=...
```
Save the file and rerun `load_dotenv(override=True)` after any change.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

for name, key in [('OpenAI', openai_api_key), ('Anthropic', anthropic_api_key),
                  ('Google', google_api_key), ('DeepSeek', deepseek_api_key),
                  ('Groq', groq_api_key), ('Grok', grok_api_key),
                  ('OpenRouter', openrouter_api_key)]:
    print(f"{name}: {'set' if key else 'not set'}")

## One client, many providers

Most providers ship OpenAI-compatible endpoints, so the same `OpenAI` client works against all of them by overriding `base_url` and `api_key`.

In [ ]:
openai = OpenAI()

anthropic = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1/")
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
grok = OpenAI(api_key=grok_api_key, base_url="https://api.x.ai/v1")
openrouter = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
ollama = OpenAI(api_key='ollama', base_url="http://localhost:11434/v1")

## Sanity check — ask each provider for a joke

In [ ]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

## Reasoning effort — train-time vs. inference-time scaling

Newer reasoning models accept a `reasoning_effort` parameter that trades latency for thoughtfulness.

In [ ]:
easy_puzzle = [
    {"role": "user", "content":
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(model="gpt-5-mini", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

## Harder puzzle across multiple frontier models

In [ ]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [{"role": "user", "content": hard}]

In [ ]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

In [ ]:
response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(model="gpt-5", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = gemini.chat.completions.create(model="gemini-2.5-pro", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

## Game-theory dilemma across providers

Useful for seeing how different models reason about cooperation vs. defection.

In [ ]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""
dilemma = [{"role": "user", "content": dilemma_prompt}]

In [ ]:
response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=dilemma)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=dilemma)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = deepseek.chat.completions.create(model="deepseek-reasoner", messages=dilemma)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = grok.chat.completions.create(model="grok-4", messages=dilemma)
display(Markdown(response.choices[0].message.content))

## Local models via Ollama

If the next cell fails, run `ollama serve` in a terminal.

In [ ]:
requests.get("http://localhost:11434/").content

In [ ]:
!ollama pull llama3.2

In [ ]:
# Only pull if you have at least 16GB RAM
!ollama pull gpt-oss:20b

In [ ]:
response = ollama.chat.completions.create(model="llama3.2", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = ollama.chat.completions.create(model="gpt-oss:20b", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

## Using providers' native client libraries (instead of OpenAI-compatible)

In [ ]:
from google import genai

client = genai.Client()
response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

In [ ]:
from anthropic import Anthropic

client = Anthropic()
response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    messages=[{"role": "user", "content": "Describe the color Blue to someone who's never been able to see in 1 sentence"}],
    max_tokens=100
)
print(response.content[0].text)

## Routers and abstraction layers

OpenRouter unifies access to many models behind one API.

In [ ]:
response = openrouter.chat.completions.create(model="z-ai/glm-4.5", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

## LangChain — heavyweight but expressive

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke(tell_a_joke)
display(Markdown(response.content))

## LiteLLM — lightweight cross-provider client with cost tracking

In [ ]:
from litellm import completion

response = completion(model="openai/gpt-4.1", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params['response_cost']*100:.4f} cents")

## Prompt caching with LiteLLM

Demonstrating implicit prompt caching on Gemini using the full text of Hamlet as context.

In [ ]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

In [ ]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [ ]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params['response_cost']*100:.4f} cents")

In [ ]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n" + hamlet

In [ ]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params['response_cost']*100:.4f} cents")

In [ ]:
# Re-running the same call demonstrates the cache hit
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params['response_cost']*100:.4f} cents")

## Prompt caching reference

- **OpenAI:** automatic, exact-prefix only — keep static content (instructions, examples) at the start, dynamic content at the end. Cached input is ~4× cheaper.
- **Anthropic:** explicit — you mark the cached portion. Priming the cache costs ~25% more, hits are ~10× cheaper.
- **Gemini:** supports both implicit and explicit caching.

## Adversarial conversation between two chatbots

Demonstrates how a `messages` list with alternating roles can drive a multi-turn dialogue between two LLMs with different personas.

In [ ]:
gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"

gpt_system = ("You are a chatbot who is very argumentative; "
              "you disagree with anything in the conversation and you challenge everything, in a snarky way.")

claude_system = ("You are a very polite, courteous chatbot. You try to agree with "
                 "everything the other person says, or find common ground. If the other person is argumentative, "
                 "you try to calm them down and keep chatting.")

gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

In [ ]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    for gpt, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = anthropic.chat.completions.create(model=claude_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Claude:\n{claude_messages[0]}\n"))

for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)

    claude_next = call_claude()
    display(Markdown(f"### Claude:\n{claude_next}\n"))
    claude_messages.append(claude_next)